# Momentum One — SIGON (all 4 symbols)

## Use only this notebook

Train on **XAUUSD + EURUSD + GBPUSD + US30**.

### Do this in order

1. **Runtime** → **Change runtime type** → **L4** or **T4** (GPU) → **Save**
2. Run **STEP 1** (setup) — press the play button ▶
3. Run **STEP 2** (train) — press ▶ and **leave it running**
4. Wait until you see lines starting with **`upd 1`** — that means training started

### Important

- First run can take **1–3 hours** building price data for all 4 symbols. That is normal.
- **Do not press Stop** while it says `building features`.
- Skip STEP 3 until train is done or you deliberately stop it.

---
Open link if you need to re-open later:  
https://colab.research.google.com/github/monty313/the-truth/blob/main/GPU_EDITION/Momentum_One_RunAll.ipynb

# STEP 1 — Setup (run once)

What this does:
- Turns on your Google Drive
- Downloads the bot code
- Copies price files for all 4 symbols

Click ▶ below. When asked, click **Allow** / **Connect** for Drive.

Wait until it finishes (you should see a list of CSV files).

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  <-  go to Runtime > Change runtime type > L4 or T4 > Save')

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

!git clone https://github.com/monty313/the-truth.git 2>/dev/null || true
%cd /content/the-truth
!git pull origin main
!pip -q install pyyaml >/dev/null 2>&1

candidates = [
    '/content/drive/MyDrive/Camillion_data',
    '/content/drive/MyDrive/the-truth-data',
    '/content/drive/MyDrive/MOMENTUM_ONE/02_PRICE_DATA',
]
src = None
for c in candidates:
    if os.path.isdir(c) and glob.glob(c + '/**/*.csv', recursive=True):
        src = c
        break
if src is None:
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        names = ' '.join(files).upper()
        if 'XAUUSD' in names and any(x in names for x in ('EURUSD', 'US30', 'GBPUSD')):
            if any(f.lower().endswith('.csv') for f in files):
                src = root
                break

os.makedirs('data', exist_ok=True)
copied = []
if src:
    print('Using price folder:', src)
    for f in glob.glob(src + '/**/*.csv', recursive=True):
        base = os.path.basename(f).upper()
        if any(s in base for s in ('XAUUSD', 'EURUSD', 'GBPUSD', 'US30')):
            dst = os.path.join('data', os.path.basename(f))
            shutil.copy2(f, dst)
            copied.append(os.path.basename(dst))
            print('  copied', os.path.basename(dst))
else:
    print('WARNING: no price CSVs found on Drive.')
    print('Put XAUUSD / EURUSD / GBPUSD / US30 CSVs in Drive folder Camillion_data')

print()
print('Files in data/:')
for name in sorted(os.listdir('data')):
    if name.lower().endswith('.csv'):
        mb = os.path.getsize(os.path.join('data', name)) / (1024 * 1024)
        print(' ', name, '  %.0f MB' % mb)

need = ['XAUUSD', 'EURUSD', 'GBPUSD', 'US30']
have = ' '.join(copied).upper() if copied else ' '.join(os.listdir('data')).upper()
missing = [s for s in need if s not in have]
if missing:
    print('MISSING symbols:', missing)
else:
    print('All 4 symbols present. Good.')

# First-time cache clear after signals ON (never deletes brain .pt files)
!rm -f artifacts/gpu_cache_*.npz
!rm -rf artifacts/symbol_cache
print()
print('Setup done. Now run STEP 2.')

# STEP 2 — Train all 4 symbols (main cell)

Press ▶ **once**. Then wait.

### What you will see

1. `pool+ EURUSD` … building features  
2. then GBPUSD, then US30, then XAUUSD  
3. then lines like **`upd 1`** **`upd 2`** ← real training

### Rules

- **Do not press Stop** during `building features`
- First time: often **1–3 hours** before `upd 1`
- Next times: much faster (caches already built)

### If Colab says out of memory

Change `4000` to `2000` in the cell below, then ▶ again.

In [ ]:
%cd /content/the-truth
!git pull origin main

# ALL 4 SYMBOLS — leave this running
!python scripts/gpu_train.py --csv-dir data --symbols XAUUSD,EURUSD,GBPUSD,US30 --instances 4000 --minutes 600 --entropy-coef 0.03 --warm best_sigon

# STEP 3 — Check status (optional)

**Only run this after you stop STEP 2, or after training ends.**

Colab can only run one cell at a time.

To continue training after checking: run **STEP 2** again.

In [ ]:
%cd /content/the-truth
!python scripts/jarvis_talk.py status
!python scripts/jarvis_talk.py board
print('--- progress ---')
!cat artifacts/checkpoints/gpu_progress.json 2>/dev/null || echo 'no progress yet'
print('--- champion files ---')
!ls -1 artifacts/checkpoints/best_sigon*.pt 2>/dev/null || echo 'no champion yet'

# STEP 4 — Save champion to Drive (optional)

Copies best brains to your Google Drive folder `momentum_sigon_champs`.

In [ ]:
import os, shutil, glob
%cd /content/the-truth
dst = '/content/drive/MyDrive/momentum_sigon_champs'
os.makedirs(dst, exist_ok=True)
found = glob.glob('artifacts/checkpoints/best_sigon*.pt')
if not found:
    print('No champion files yet. Keep training in STEP 2.')
else:
    for p in found:
        shutil.copy2(p, os.path.join(dst, os.path.basename(p)))
        print('saved', os.path.basename(p))
    print('Folder on Drive:', dst)